# 3. Numerical Integration (Quadrature)

Approximate $\int_a^b f(x)\,dx$ using function evaluations. This notebook covers:
- **Trapezoidal rule** (composite)
- **Simpson's rule** (composite)
- **Gauss-Legendre quadrature**
- Error analysis and convergence orders

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf

%matplotlib inline

# Test function: f(x) = exp(-x^2) on [0, 2]
f = lambda x: np.exp(-x**2)
a, b = 0, 2
exact = np.sqrt(np.pi) / 2 * erf(2)
print(f"Exact integral: {exact:.12f}")

x_plot = np.linspace(a, b, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_plot, f(x_plot), 'b-', lw=2)
ax.fill_between(x_plot, f(x_plot), alpha=0.2, color='steelblue')
ax.set_xlabel('x')
ax.set_ylabel('$f(x) = e^{-x^2}$')
ax.set_title('Area under the curve')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.2 Composite Trapezoidal Rule

$$\int_a^b f(x)\,dx \approx \frac{h}{2}\left[f(a) + 2\sum_{i=1}^{n-1}f(x_i) + f(b)\right]$$

Error: $O(h^2)$ where $h = (b-a)/n$.

In [ ]:
def trapezoidal(f, a, b, n):
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h * (y[0]/2 + np.sum(y[1:-1]) + y[-1]/2)

for n in [4, 8, 16, 32, 64, 128]:
    approx = trapezoidal(f, a, b, n)
    error = abs(approx - exact)
    print(f"n = {n:>4d}: I = {approx:.10f}, error = {error:.2e}")

## 3.3 Composite Simpson's Rule

$$\int_a^b f(x)\,dx \approx \frac{h}{3}\left[f(a) + 4\sum_{\text{odd}} f(x_i) + 2\sum_{\text{even}} f(x_i) + f(b)\right]$$

Error: $O(h^4)$, requires $n$ even.

In [ ]:
def simpson(f, a, b, n):
    if n % 2 != 0:
        n += 1
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h / 3 * (y[0] + 4 * np.sum(y[1:-1:2]) + 2 * np.sum(y[2:-2:2]) + y[-1])

for n in [4, 8, 16, 32, 64, 128]:
    approx = simpson(f, a, b, n)
    error = abs(approx - exact)
    print(f"n = {n:>4d}: I = {approx:.10f}, error = {error:.2e}")

## 3.4 Gauss-Legendre Quadrature

Uses optimally-chosen nodes and weights. Exact for polynomials of degree up to $2n-1$:
$$\int_{-1}^{1} f(x)\,dx \approx \sum_{i=1}^{n} w_i f(x_i)$$

In [ ]:
def gauss_legendre(f, a, b, n):
    nodes, weights = np.polynomial.legendre.leggauss(n)
    x = 0.5 * (b - a) * nodes + 0.5 * (b + a)
    w = 0.5 * (b - a) * weights
    return np.sum(w * f(x))

for n in [2, 3, 4, 5, 8, 12]:
    approx = gauss_legendre(f, a, b, n)
    error = abs(approx - exact)
    print(f"n = {n:>2d} points: I = {approx:.12f}, error = {error:.2e}")

In [ ]:
# Convergence comparison
ns = np.array([4, 8, 16, 32, 64, 128, 256])
err_trap = [abs(trapezoidal(f, a, b, n) - exact) for n in ns]
err_simp = [abs(simpson(f, a, b, n) - exact) for n in ns]
err_gauss = [abs(gauss_legendre(f, a, b, n) - exact) for n in [2, 3, 4, 5, 6, 7, 8]]

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(ns, err_trap, 'o-', label='Trapezoidal ($O(h^2)$)', markersize=5)
ax.loglog(ns, err_simp, 's-', label="Simpson's ($O(h^4)$)", markersize=5)
ax.loglog([2, 3, 4, 5, 6, 7, 8], err_gauss, '^-', label='Gauss-Legendre', markersize=5)
h_ref = np.array(ns, dtype=float)
ax.loglog(ns, 0.5 * (ns[0]/h_ref)**2 * err_trap[0], 'k--', alpha=0.3, label='$O(n^{-2})$')
ax.loglog(ns, 0.5 * (ns[0]/h_ref)**4 * err_simp[0], 'k:', alpha=0.3, label='$O(n^{-4})$')
ax.set_xlabel('Number of subintervals / points')
ax.set_ylabel('Absolute error')
ax.set_title('Convergence of Quadrature Methods')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

| Method | Order | Best for |
|--------|-------|----------|
| Trapezoidal | $O(h^2)$ | Simple functions, adaptive refinement |
| Simpson | $O(h^4)$ | Smooth functions, moderate accuracy |
| Gauss-Legendre | $O(h^{2n})$ | High accuracy with few evaluations |

- Gauss quadrature achieves much higher accuracy with fewer function evaluations
- For non-smooth functions, adaptive methods (subdivide where error is large) are preferred
- In practice, use `scipy.integrate.quad` which combines adaptive strategies